#MOSAIC Fusion Head Training


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

from fusionmlp import FusionMLP
from fusion_dataset import get_dataloaders

*Initialsing Model*

---
Initialising the model parameters

In [ ]:
# Init model
model = FusionMLP()
criterion = BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=0.001)

# Initialize dataloaders correctly with paths to your train, val, and test splits
loaders = get_dataloaders(
    train_csv='train_set_rtf_scores.csv', 
    train_parquet='train_gw_map.parquet', # update path as needed
    val_csv='validation_set_rtf_scores.csv', 
    val_parquet='val_gw_map.parquet',       # update path as needed
    test_csv='test_set_rtf_scores.csv', 
    test_parquet='test_gw_map.parquet',     # update path as needed
    batch_size=64
)

*Training the model*

In [ ]:
# Train
epochs = 30
for i in range(epochs):
    model.train()
    for batch_x, batch_y in loaders['train']:
        optimizer.zero_grad()
        outputs = model(batch_x)
        outputs = outputs.squeeze(-1)
        batch_y = batch_y.float()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    if (i+1) % 10 == 0:
        print(f"Epoch {i+1}: {loss.item():.6f}")


*Saving Updated Model*

In [ ]:
# Save
torch.save(model.state_dict(), 'fusion_mlp.pt')
print("Model saved.")

FusionMLP(
  (layer1): Linear(in_features=3, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=16, bias=True)
  (layer3): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
)


*Evaluating the Model*

In [ ]:
# Evaluate
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch_x, batch_y in loaders['test']:
        preds = torch.sigmoid(model(batch_x).squeeze(-1))
        all_preds.extend(preds.numpy())
        all_labels.extend(batch_y.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

fpr, tpr, thresholds = roc_curve(all_labels, all_preds)
roc_auc = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(all_labels, all_preds)
ap = average_precision_score(all_labels, all_preds)

print(f"\nTest AUC: {roc_auc:.4f}")
print(f"Avg Precision: {ap:.4f}")

pred_binary = (all_preds >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, pred_binary).ravel()
print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")